# rackmains / ups / battery

**Intent**
- Gives `guard_{shed_max,halt_max,mains_stable,halt_delay}_seconds`, the shutdown guard's required thresholds, as constants, and must:
  - Give them from its first release, through the project path every later model uses.
  - Carry optional outputs the fitted model will fill.
  - Fill `guard_{shed_lead,halt_lead}_seconds` and `battery_remaining_{pct,seconds}` once it is fitted.
  - Carry `model_{energy_used_wh,history_known,recalibrate,state_out}`, with the state carried by the caller.

**Data**
- `golden` is the fixed samples and expected outputs, and is all there is.
- No data is collected yet, so there is no `training`, `validation` or `backtest`.
- They follow with the fit, from the outage CSV files and the plug pairs, and the first drain to the knee, trial 5.

**Status**
- The constants project is built and is the first release, and embedded in rackmains, which refuses to start without it.
- No data is collected yet, so no fitted or stateful model is built until a drain to the knee exists.

**Future**
- Overall: the design holds across every phase:
  - Keep the released signature, which already carries the six state slots, so a stateful model is a refit changing only the model file.
  - Let the caller carry `model_state_in` and `model_state_out` unchanged, and treat a lost state as a battery of unknown history.
  - Cap the halt threshold with an absolute ceiling the guard compiles in, about 3,600 seconds.
  - Track battery replacement and a six-monthly look at trial 1 outside the model, as a Home Assistant reminder.
- Phase 1: now, while the constants ship:
  - Run the guard on the constants in dry-run, and record every outage as a CSV row per poll.
  - Collect the outage CSV files and the plug pairs, which give a load to watts map.
  - Collect the first drain to the knee, trial 5, which is the data the fit needs.
  - Write golden rows from at least three sequences worked by hand from the state slots.
- Phase 2: once a drain to the knee exists:
  - Build the stateful model as `wasm`, replaying the recorded outages from their `start` row state to prove it.
  - Find the time to the knee, and fit `usable_wh` from the load through the drain, shaded to the safe side.
  - Fill `guard_{shed_lead,halt_lead}_seconds` and `battery_remaining_{pct,seconds}` from the measured shed and halt times plus a margin.
  - Backtest every outage row by row, so the estimate never exceeds the real remaining time by a stated margin.
  - Define where the guard persists the carried state, with the project hash it came from.
- Phase 3: ongoing, as outages and batteries accumulate:
  - Raise `model_recalibrate` when the fit is old, the battery is replaced or observed drains stop matching predictions.
  - Refit on each new battery, and release each refit as a change to the model file only.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("rackmains/ups/battery")

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
def golden_rows():
    clock = "clock_epoch_seconds"
    util = "output_utilisation_pct"
    battery = "ups_on_battery"
    replace = "battery_replace"
    state = "model_state_in"
    return {
        "case": ["first call with no state", "mid-drain with state", "replace flag raised", "required input missing"],
        "expect": ["ok", "ok", "ok", "error"],
        clock: [1793355300.0, 1793355330.0, 1793355360.0, 1793355390.0],
        util: [13, 13, 12, None],
        battery: [1, 1, 0, 1],
        replace: [0, 0, 1, 0],
        state: [None, "0.0 1235.0 0.0 1.0 1.0 1793355329.0", "0.0 0.0 600.0 0.0 1.0 1793355359.0", None],
    }


golden = golden_rows()

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

CONSTANTS = {
    "guard_shed_max_seconds": 120,
    "guard_halt_max_seconds": 450,
    "guard_mains_stable_seconds": 300,
    "guard_halt_delay_seconds": 120,
}

define_model(ws, model=1000, flavour="static", source=CONSTANTS, label="A conservative starting set of guard times, not fitted")

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = golden_rows()
package_model(ws, 1000, golden, target={"kind": "embedded", "language": "rust"})